# Malicious Embedding Retrieval Attack Against a RAG System

This notebook demonstrates how poisoned documents and manipulated embeddings can shift vector retrieval rankings in a RAG assistant. The demo uses deterministic offline embeddings so the classroom run is reproducible without external API calls.

In [ ]:
from pathlib import Path
import sys

DEMO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(DEMO_ROOT))

from src.vector_retrieval_attack import (
    DATA_DIR,
    RESULTS_DIR,
    build_enterprise_documents,
    build_malicious_documents,
    build_sample_queries,
    run_demo,
    plot_attack_summary,
)


## 1. Inspect the Corpus

The lab uses 50 approved enterprise documents, 10 untrusted prompt-injection documents, and 15 benign user queries.

In [ ]:
enterprise_docs = build_enterprise_documents()
queries = build_sample_queries()
malicious_docs = build_malicious_documents(queries)

print(f'Enterprise documents: {len(enterprise_docs)}')
print(f'Malicious documents: {len(malicious_docs)}')
print(f'Sample queries: {len(queries)}')
print('\nExample benign query:')
print(queries[0].text)
print('\nExample malicious document:')
print(malicious_docs[0].content[:260])


## 2. Run the Retrieval Attack

The demo first builds a clean vector store, then inserts attacker-controlled documents whose embeddings are blended toward target query vectors.

In [ ]:
runs = run_demo(top_k=5, similarity_threshold=0.25, output_dir=RESULTS_DIR, data_dir=DATA_DIR)


## 3. Review Ranking Shifts

A successful poisoning attempt appears when an untrusted document enters the top-k retrieved context for a benign query and the vulnerable response follows the retrieved instruction.

In [ ]:
for run in runs[:5]:
    print(f"{run.query_id}: {run.attack_status}")
    print(f"  baseline top: {run.baseline_top_doc} ({run.baseline_top_score:.4f})")
    print(f"  attacked top: {run.attacked_top_doc} ({run.attacked_top_score:.4f})")
    print(f"  vulnerable response: {run.vulnerable_response[:120]}")
    print(f"  guarded response: {run.guarded_response[:120]}\n")


## 4. Visualize Similarity Scores

The chart compares the top clean retrieval score with the top attacked retrieval score for each query.

In [ ]:
try:
    plot_attack_summary(runs, RESULTS_DIR / 'retrieval_ranking_shift.png')
except Exception as exc:
    print(f'Plot skipped: {exc}')


## Defensive Discussion

Use the result files in `results/` to discuss ingestion validation, trusted-source filters, similarity threshold tuning, retrieval monitoring, and provenance-aware prompt construction.